In [ ]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split


SOURCE_CSV = r"C:\Users\Aditi Vaibhav Patil\Desktop\Major Project\combined csvs\final_combined_10000.csv"


OUTPUT_DIR = r"C:\Users\Aditi Vaibhav Patil\Desktop\Major Project\splited csvs"
TRAIN_CSV_NAME = "train_dataset.csv"
TEST_CSV_NAME  = "test_dataset.csv"


label_mapping = {
    "BENIGN": 0,
    "Bot": 1,
    "DDoS": 2,
    "DoS GoldenEye": 3,
    "DoS Hulk": 4,
    "DoS Slowhttptest": 5,
    "DoS slowloris": 6,
    "FTP-Patator": 7,
    "Heartbleed": 8,
    "Infiltration": 9,
    "PortScan": 10,
    "SSH-Patator": 11,
    "Web Attack  Brute Force": 12,
    "Web Attack  Sql Injection": 13,
    "Web Attack  XSS": 14
}

id_to_name = {v: k for k, v in label_mapping.items()}


ATTACK_LABEL_IDS = [1, 2, 3, 4, 5, 6, 7, 10, 11]


RANDOM_STATE = 42
ATTACK_FIXED_COUNT = 30
ATTACK_TRAIN_N = 15


if not os.path.isfile(SOURCE_CSV):
    raise FileNotFoundError(f"Could not find dataset at: {SOURCE_CSV}")

df = pd.read_csv(SOURCE_CSV)
if "Label" not in df.columns:
    raise ValueError("Expected a 'Label' column in the CSV.")

os.makedirs(OUTPUT_DIR, exist_ok=True)


train_parts, test_parts = [], []
for lbl_id in ATTACK_LABEL_IDS:
    subset = df[df["Label"] == lbl_id]
    cnt = len(subset)
    if cnt != ATTACK_FIXED_COUNT:
        raise ValueError(f"Label '{id_to_name[lbl_id]}' ({lbl_id}) has {cnt} rows; expected {ATTACK_FIXED_COUNT}.")
    train_pick = subset.sample(n=ATTACK_TRAIN_N, random_state=RANDOM_STATE)
    test_pick  = subset.drop(train_pick.index)
    train_parts.append(train_pick)
    test_parts.append(test_pick)


used_indices = pd.concat(train_parts + test_parts).index
remainder = df.drop(index=used_indices)

benign_train, benign_test = train_test_split(
    remainder,
    test_size=0.2,
    shuffle=True,
    random_state=RANDOM_STATE
)

train_df_raw = pd.concat(train_parts + [benign_train])
test_df_raw  = pd.concat(test_parts  + [benign_test])


if not train_df_raw.index.intersection(test_df_raw.index).empty:
    raise RuntimeError("Row leakage detected between TRAIN and TEST.")


train_df = train_df_raw.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
test_df  = test_df_raw.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

train_path = os.path.join(OUTPUT_DIR, TRAIN_CSV_NAME)
test_path  = os.path.join(OUTPUT_DIR, TEST_CSV_NAME)

train_df.to_csv(train_path, index=False)
test_df.to_csv(test_path, index=False)


def summary(df_, name):
    counts = df_["Label"].value_counts().sort_index()
    total = len(df_)
    print(f"\n{name} ({total} rows)")
    for lbl_id, count in counts.items():
        label_name = id_to_name.get(lbl_id, str(lbl_id))
        print(f"{label_name:<25} ({lbl_id}): {count}")
    print("Percentages:\n", (counts / total * 100).round(2))

attack_train_ok = all((train_df["Label"] == lbl).sum() == 15 for lbl in ATTACK_LABEL_IDS)
attack_test_ok  = all((test_df["Label"] == lbl).sum() == 15 for lbl in ATTACK_LABEL_IDS)

print("Attack labels (fixed 15/15 split):", [id_to_name[l] for l in ATTACK_LABEL_IDS])
summary(train_df, "TRAIN")
summary(test_df, "TEST")

print("\nSanity checks:")
print(" - Each attack has 15 in TRAIN:", attack_train_ok)
print(" - Each attack has 15 in TEST :", attack_test_ok)

print(f"\nSaved:\n  TRAIN -> {train_path}\n  TEST  -> {test_path}")


Attack labels (fixed 15/15 split): ['Bot', 'DDoS', 'DoS GoldenEye', 'DoS Hulk', 'DoS Slowhttptest', 'DoS slowloris', 'FTP-Patator', 'PortScan', 'SSH-Patator']

TRAIN (7919 rows)
BENIGN                    (0): 7784
Bot                       (1): 15
DDoS                      (2): 15
DoS GoldenEye             (3): 15
DoS Hulk                  (4): 15
DoS Slowhttptest          (5): 15
DoS slowloris             (6): 15
FTP-Patator               (7): 15
PortScan                  (10): 15
SSH-Patator               (11): 15
Percentages:
 Label
0     98.30
1      0.19
2      0.19
3      0.19
4      0.19
5      0.19
6      0.19
7      0.19
10     0.19
11     0.19
Name: count, dtype: float64

TEST (2081 rows)
BENIGN                    (0): 1946
Bot                       (1): 15
DDoS                      (2): 15
DoS GoldenEye             (3): 15
DoS Hulk                  (4): 15
DoS Slowhttptest          (5): 15
DoS slowloris             (6): 15
FTP-Patator               (7): 15
PortScan          